In [8]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchtext.vocab import Vocab
from torchtext.data.utils import get_tokenizer
from collections import Counter
import spacy

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define paths to the data files
data_folder = '/content/drive/MyDrive/datasets/manipuri MT/English-Manipuri/parallel'
train_src_file = os.path.join(data_folder, 'en-mni-train-mni.txt')
train_tgt_file = os.path.join(data_folder, 'en-mni-train-en.txt')
valid_src_file = os.path.join(data_folder, 'en-mni-valid-mni.txt')
valid_tgt_file = os.path.join(data_folder, 'en-mni-valid-en.txt')
test_src_file = os.path.join(data_folder, 'en-mni-test-mni.txt')
test_tgt_file = os.path.join(data_folder, 'en-mni-test-en.txt')

In [11]:
from collections import Counter

# Load the training data
train_src, train_tgt = load_sentences(train_src_file, train_tgt_file)

# Build the source vocabulary
src_tokens = [token for sentence in train_src for token in sentence.split()]
src_vocab = Vocab(Counter(src_tokens), min_freq=2, specials=['<pad>', '<sos>', '<eos>'])

# Build the target vocabulary
tgt_tokens = [token for sentence in train_tgt for token in sentence.split()]
tgt_vocab = Vocab(Counter(tgt_tokens), min_freq=2, specials=['<pad>', '<sos>', '<eos>'])


NameError: name 'load_sentences' is not defined

In [5]:
# Define the dataset class
class MTDataset(Dataset):
    def __init__(self, src_file, tgt_file, src_vocab, tgt_vocab):
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.src_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')
        self.tgt_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')

        with open(src_file, 'r') as f:
            self.src_sentences = f.readlines()
        with open(tgt_file, 'r') as f:
            self.tgt_sentences = f.readlines()

        assert len(self.src_sentences) == len(self.tgt_sentences)

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx].strip()
        tgt_sentence = self.tgt_sentences[idx].strip()

        src_tokens = self.src_tokenizer(src_sentence)
        tgt_tokens = self.tgt_tokenizer(tgt_sentence)

        src_ids = [self.src_vocab[token] for token in src_tokens]
        tgt_ids = [self.tgt_vocab[token] for token in tgt_tokens]

        return torch.tensor(src_ids), torch.tensor(tgt_ids)


In [6]:
# Define the BiLSTM model
class BiLSTMTranslator(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(BiLSTMTranslator, self).__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, embedding_dim)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True)
        self.decoder = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        max_len = tgt.size(1)
        tgt_vocab_size = self.fc.out_features

        outputs = torch.zeros(batch_size, max_len, tgt_vocab_size).to(src.device)

        encoder_outputs, (hidden, cell) = self.encoder(self.src_embedding(src))

        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        cell = self.dropout(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1))

        decoder_input = tgt[:, 0].unsqueeze(1)

        for t in range(1, max_len):
            decoder_output, (hidden, cell) = self.decoder(self.src_embedding(decoder_input), (hidden, cell))
            output = self.fc(self.dropout(decoder_output))
            outputs[:, t] = output.squeeze(1)

            top1 = output.argmax(1)
            decoder_input = tgt[:, t].unsqueeze(1) if random.random() < teacher_forcing_ratio else top1.unsqueeze(1)

        return outputs


In [7]:
# Define the training loop
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        loss = criterion(output[:, 1:].reshape(-1, output.size(2)), tgt[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


In [9]:
# Define the validation loop
def validate(model, valid_loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in valid_loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt)
            loss = criterion(output[:, 1:].reshape(-1, output.size(2)), tgt[:, 1:].reshape(-1))
            total_loss += loss.item()

    return total_loss / len(valid_loader)


In [10]:
# Define the main training function
def main():
    # Load the data
    train_dataset = MTDataset(train_src_file, train_tgt_file, src_vocab, tgt_vocab)
    valid_dataset = MTDataset(valid_src_file, valid_tgt_file, src_vocab, tgt_vocab)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=32)

    # Define the model, optimizer, and criterion
    model = BiLSTMTranslator(len(src_vocab), len(tgt_vocab), 256, 512, 2, 0.5).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.stoi['<pad>'])

    # Train the model
    best_valid_loss = float('inf')

    for epoch in range(10):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        valid_loss = validate(model, valid_loader, criterion, device)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), 'best_model.pt')

        print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

    # Load the best model and evaluate on the test set
    model.load_state_dict(torch.load('best_model.pt'))
    test_dataset = MTDataset(test_src_file, test_tgt_file, src_vocab, tgt_vocab)
    test_loader = DataLoader(test_dataset, batch_size=32)
    test_loss = validate(model, test_loader, criterion, device)

    print(f'Test Loss: {test_loss:.4f}')

if __name__ == '__main__':
    main()


NameError: name 'src_vocab' is not defined